In [1]:

# STEP 1: WORD EMBEDDINGS


class WordEmbedding:

    def __init__(self):
        # Example sentence: "We study NLP"
        # Each word is represented using 2 values
        self.vectors = [
            [1, 1],      # We
            [0, 1],      # study
            [1, 0]       # NLP
        ]

    def show(self):
        print("Word Embeddings:")
        for item in self.vectors:
            print(item)


words = WordEmbedding()
words.show()

Word Embeddings:
[1, 1]
[0, 1]
[1, 0]


In [2]:
# STEP 2: POSITION INFORMATION


class PositionEncoding:

    def __init__(self):
        self.position_values = [
            [0.00, 1.00],
            [0.84, 0.54],
            [0.91, -0.42]
        ]

    def encode(self, vectors):

        encoded = []

        for i in range(len(vectors)):
            new_row = []

            for j in range(len(vectors[0])):
                new_row.append(
                    vectors[i][j] + self.position_values[i][j]
                )

            encoded.append(new_row)

        return encoded

    def show(self, encoded):

        print("Embeddings with Position:")
        for row in encoded:
            print(row)


pos_encoder = PositionEncoding()

encoded_vectors = pos_encoder.encode(words.vectors)

pos_encoder.show(encoded_vectors)

Embeddings with Position:
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]


In [3]:
# STEP 3: BASIC MATRIX OPERATIONS

class MatrixCalculator:

    def multiply(self, first, second):

        rows = len(first)
        common = len(first[0])
        columns = len(second[0])

        answer = []

        for i in range(rows):

            current_row = []

            for j in range(columns):

                total = 0

                for k in range(common):
                    total += first[i][k] * second[k][j]

                current_row.append(total)

            answer.append(current_row)

        return answer

    def transpose(self, matrix):

        result = []

        for j in range(len(matrix[0])):

            row = []

            for i in range(len(matrix)):
                row.append(matrix[i][j])

            result.append(row)

        return result


calculator = MatrixCalculator()

In [4]:
# STEP 4: GENERATE QUERY, KEY AND VALUE


class AttentionVectors:

    def __init__(self):

        # Simple transformation matrices
        self.query_weights = [
            [1, 0],
            [0, 1]
        ]

        self.key_weights = [
            [1, 0],
            [0, 1]
        ]

        self.value_weights = [
            [1, 0],
            [0, 1]
        ]

    def generate(self, data):

        query = calculator.multiply(data, self.query_weights)
        key = calculator.multiply(data, self.key_weights)
        value = calculator.multiply(data, self.value_weights)

        return query, key, value


attention_vectors = AttentionVectors()

Q, K, V = attention_vectors.generate(encoded_vectors)

print("Query Matrix:")
for row in Q:
    print(row)

print("\nKey Matrix:")
for row in K:
    print(row)

print("\nValue Matrix:")
for row in V:
    print(row)

Query Matrix:
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]

Key Matrix:
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]

Value Matrix:
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]


In [5]:
# STEP 5: SCALED DOT PRODUCT

import math


class DotProductAttention:

    def calculate_scores(self, Q, K):

        # Transpose the Key matrix
        K_transpose = calculator.transpose(K)

        # Calculate Q * K^T
        scores = calculator.multiply(Q, K_transpose)

        # Number of dimensions in each key vector
        dimension = len(K[0])

        # Scale the scores
        factor = math.sqrt(dimension)

        for i in range(len(scores)):
            for j in range(len(scores[0])):
                scores[i][j] /= factor

        return scores


dot_attention = DotProductAttention()

score_matrix = dot_attention.calculate_scores(Q, K)

print("Scaled Dot Product Scores:")

for row in score_matrix:
    print(row)

Scaled Dot Product Scores:
[3.5355339059327373, 2.7718585822512662, 0.756604255869606]
[2.7718585822512662, 2.1759089870672437, 0.6771254536642378]
[0.756604255869606, 0.6771254536642378, 2.7043298846479513]


In [6]:
#
# STEP 6: SOFTMAX

class SoftmaxFunction:

    def single_row(self, row):

        exponentials = []

        for number in row:
            exponentials.append(math.exp(number))

        denominator = sum(exponentials)

        probabilities = []

        for number in exponentials:
            probabilities.append(number / denominator)

        return probabilities

    def transform(self, scores):

        result = []

        for row in scores:
            result.append(self.single_row(row))

        return result


softmax_layer = SoftmaxFunction()

weights = softmax_layer.transform(score_matrix)

print("Attention Probabilities:")

for row in weights:
    print(row)

Attention Probabilities:
[0.6544264051497775, 0.3049304784023443, 0.04064311644787826]
[0.597320844481012, 0.3291471126671976, 0.07353204285179042]
[0.11190291751838921, 0.10335326582842719, 0.7847438166531835]


In [7]:
# ============================================
# STEP 7: FINAL ATTENTION OUTPUT
# ============================================

class WeightedOutput:

    def generate(self, weights, values):

        return calculator.multiply(weights, values)


output_layer = WeightedOutput()

final_result = output_layer.generate(weights, V)

print("Final Attention Result:")

for row in final_result:
    print(row)

Final Attention Result:
[0.9881963594231942, 1.7613756381310561]
[1.0142506209683777, 1.6706447844717562]
[1.6975803506218485, 0.05337746141821925]


In [8]:
# ============================================
# COMPLETE SELF-ATTENTION ENCODER
# ============================================

class SimpleEncoder:

    def __init__(self):

        self.embedding = WordEmbedding()
        self.position = PositionEncoding()
        self.qkv = AttentionVectors()
        self.attention = DotProductAttention()
        self.softmax = SoftmaxFunction()
        self.output = WeightedOutput()

    def execute(self):

        # 1. Get word vectors
        X = self.embedding.vectors

        print("----- STEP 1: EMBEDDINGS -----")
        for row in X:
            print(row)

        # 2. Add positional information
        encoded = self.position.encode(X)

        print("\n----- STEP 2: POSITION ENCODING -----")
        for row in encoded:
            print(row)

        # 3. Create Q, K and V
        Q, K, V = self.qkv.generate(encoded)

        print("\n----- STEP 3: Q -----")
        for row in Q:
            print(row)

        print("\n----- K -----")
        for row in K:
            print(row)

        print("\n----- V -----")
        for row in V:
            print(row)

        # 4. Calculate scaled attention
        scores = self.attention.calculate_scores(Q, K)

        print("\n----- STEP 4: SCALED SCORES -----")
        for row in scores:
            print(row)

        # 5. Convert scores into probabilities
        weights = self.softmax.transform(scores)

        print("\n----- STEP 5: ATTENTION WEIGHTS -----")
        for row in weights:
            print(row)

        # 6. Generate final output
        result = self.output.generate(weights, V)

        print("\n----- STEP 6: FINAL OUTPUT -----")
        for row in result:
            print(row)


model = SimpleEncoder()
model.execute()

----- STEP 1: EMBEDDINGS -----
[1, 1]
[0, 1]
[1, 0]

----- STEP 2: POSITION ENCODING -----
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]

----- STEP 3: Q -----
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]

----- K -----
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]

----- V -----
[1.0, 2.0]
[0.84, 1.54]
[1.9100000000000001, -0.42]

----- STEP 4: SCALED SCORES -----
[3.5355339059327373, 2.7718585822512662, 0.756604255869606]
[2.7718585822512662, 2.1759089870672437, 0.6771254536642378]
[0.756604255869606, 0.6771254536642378, 2.7043298846479513]

----- STEP 5: ATTENTION WEIGHTS -----
[0.6544264051497775, 0.3049304784023443, 0.04064311644787826]
[0.597320844481012, 0.3291471126671976, 0.07353204285179042]
[0.11190291751838921, 0.10335326582842719, 0.7847438166531835]

----- STEP 6: FINAL OUTPUT -----
[0.9881963594231942, 1.7613756381310561]
[1.0142506209683777, 1.6706447844717562]
[1.6975803506218485, 0.05337746141821925]
